In [1]:
import pandas as pd
import numpy as np
import plotly.express as px

In [2]:
df = pd.read_excel("UrbanCartSuperstoreClean Data.xlsx")

In [3]:
df.columns = df.columns.str.strip()

In [4]:
print("Shape:", df.shape)

Shape: (9994, 25)


In [5]:
print(df.head())

   Row ID        Order ID Order Date  Ship Date  Delivery Days  \
0       1  CA-2016-152156 2016-11-08 2016-11-11              3   
1       2  CA-2016-152156 2016-11-08 2016-11-11              3   
2       3  CA-2016-138688 2016-06-12 2016-06-16              4   
3       4  US-2015-108966 2015-10-11 2015-10-18              7   
4       5  US-2015-108966 2015-10-11 2015-10-18              7   

        Ship Mode Customer ID    Customer Name    Segment        Country  ...  \
0    Second Class    CG-12520      Claire Gute   Consumer  United States  ...   
1    Second Class    CG-12520      Claire Gute   Consumer  United States  ...   
2    Second Class    DV-13045  Darrin Van Huff  Corporate  United States  ...   
3  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States  ...   
4  Standard Class    SO-20335   Sean O'Donnell   Consumer  United States  ...   

          Category Sub-Category  \
0        Furniture    Bookcases   
1        Furniture       Chairs   
2  Office S

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 9994 entries, 0 to 9993
Data columns (total 25 columns):
 #   Column         Non-Null Count  Dtype         
---  ------         --------------  -----         
 0   Row ID         9994 non-null   int64         
 1   Order ID       9994 non-null   object        
 2   Order Date     9994 non-null   datetime64[ns]
 3   Ship Date      9994 non-null   datetime64[ns]
 4   Delivery Days  9994 non-null   int64         
 5   Ship Mode      9994 non-null   object        
 6   Customer ID    9994 non-null   object        
 7   Customer Name  9994 non-null   object        
 8   Segment        9994 non-null   object        
 9   Country        9994 non-null   object        
 10  City           9994 non-null   object        
 11  State          9994 non-null   object        
 12  Postal Code    9994 non-null   int64         
 13  Region         9994 non-null   object        
 14  Product ID     9994 non-null   object        
 15  Category       9994 n

In [7]:
print("Missing values:\n", df.isnull().sum())

Missing values:
 Row ID           0
Order ID         0
Order Date       0
Ship Date        0
Delivery Days    0
Ship Mode        0
Customer ID      0
Customer Name    0
Segment          0
Country          0
City             0
State            0
Postal Code      0
Region           0
Product ID       0
Category         0
Sub-Category     0
Product Name     0
Sales            0
Quantity         0
Discount         0
Profit           0
Profit Margin    0
Months           0
Years            0
dtype: int64


In [8]:
print("Duplicate rows:", df.duplicated().sum())

Duplicate rows: 0


In [9]:
# Convert text money/percent columns to numbers
for col in ["Sales", "Profit"]:
    df[col] = (df[col].astype(str)
                      .str.replace(r"[$,\s]", "", regex=True)
                      .astype(float))

df["Profit Margin"] = (df["Profit Margin"].astype(str)
                       .str.replace("%", "", regex=False)
                       .astype(float))          # now in percent, e.g. 16.0

# Convert dates
df["Order Date"] = pd.to_datetime(df["Order Date"], dayfirst=True, errors="coerce")
df["Ship Date"] = pd.to_datetime(df["Ship Date"], dayfirst=True, errors="coerce")


# Recalculate time and delivery columns so they are reliable
df["Year"] = df["Order Date"].dt.year
df["Month"] = df["Order Date"].dt.month
df["Month_Name"] = df["Order Date"].dt.month_name()
df["Delivery_Days"] = (df["Ship Date"] - df["Order Date"]).dt.days

print(df[["Sales", "Profit", "Quantity", "Discount"]].describe())

              Sales       Profit     Quantity     Discount
count   9994.000000  9994.000000  9994.000000  9994.000000
mean     229.858001    28.656896     3.789574     0.156203
std      623.245101   234.260108     2.225110     0.206452
min        0.444000 -6599.978000     1.000000     0.000000
25%       17.280000     1.728750     2.000000     0.000000
50%       54.490000     8.666500     3.000000     0.200000
75%      209.940000    29.364000     5.000000     0.200000
max    22638.480000  8399.976000    14.000000     0.800000


In [10]:
total_sales = df["Sales"].sum()
total_profit = df["Profit"].sum()

print("========== OVERALL SUMMARY ==========")
print("Total Sales: $", round(total_sales, 2))
print("Total Profit: $", round(total_profit, 2))
print("Overall Profit Margin: ", round(total_profit / total_sales * 100, 2), "%")
print("Total Orders:", df["Order ID"].nunique())
print("Total Customers:", df["Customer ID"].nunique())
print("Total Quantity:", df["Quantity"].sum())

========== OVERALL SUMMARY ==========
Total Sales: $ 2297200.86
Total Profit: $ 286397.02
Overall Profit Margin:  12.47 %
Total Orders: 5009
Total Customers: 793
Total Quantity: 37873


In [12]:
region = (df.groupby("Region")
            .agg(Sales=("Sales", "sum"),
                 Profit=("Profit", "sum"),
                 Orders=("Order ID", "nunique"),
                 Quantity=("Quantity", "sum"),
                 Avg_Discount=("Discount", "mean"))
            .reset_index())

region["Profit_Margin_%"] = (region["Profit"] / region["Sales"] * 100).round(2)
region["Sales_Share_%"] = (region["Sales"] / region["Sales"].sum() * 100).round(2)
region["Avg_Order_Value"] = (region["Sales"] / region["Orders"]).round(2)
region = region.sort_values("Sales", ascending=False)
print(region.round(2))

    Region      Sales     Profit  Orders  Quantity  Avg_Discount  \
3     West  725457.82  108418.45    1611     12266          0.11   
1     East  678781.24   91522.78    1401     10618          0.15   
0  Central  501239.89   39706.36    1175      8780          0.24   
2    South  391721.90   46749.43     822      6209          0.15   

   Profit_Margin_%  Sales_Share_%  Avg_Order_Value  
3            14.94          31.58           450.32  
1            13.48          29.55           484.50  
0             7.92          21.82           426.59  
2            11.93          17.05           476.55  


In [13]:
px.bar(region, x="Region", y="Sales", text_auto=".2s",
       title="Total Sales by Region").show()

In [14]:
px.bar(region, x="Region", y="Profit", text_auto=".2s",
       title="Total Profit by Region").show()

In [15]:
px.pie(region, names="Region", values="Sales", hole=0.4,
       title="Sales Share by Region").show()

In [16]:
# Sales vs profit side by side
px.bar(region.melt(id_vars="Region", value_vars=["Sales", "Profit"]),
       x="Region", y="value", color="variable", barmode="group",
       title="Sales vs Profit by Region").show()

In [18]:
# Heatmap of sales by region and sub-category
pivot = df.pivot_table(index="Sub-Category", columns="Region",
                       values="Profit", aggfunc="sum")
px.imshow(pivot, text_auto=".0f", aspect="auto",
          title="Profit by Sub-Category and Region").show()

In [22]:
num = df[["Sales", "Quantity", "Discount", "Profit", "Profit Margin", "Delivery_Days"]]
px.imshow(num.corr().round(2), text_auto=".2f",
          title="Correlation Heatmap").show()

In [23]:
best_sales = region.iloc[0]
best_profit = region.sort_values("Profit", ascending=False).iloc[0]
best_margin = region.sort_values("Profit_Margin_%", ascending=False).iloc[0]
worst_margin = region.sort_values("Profit_Margin_%").iloc[0]

print("========== REGIONAL PERFORMANCE SUMMARY ==========")
print("Highest sales region:", best_sales["Region"], "($", round(best_sales["Sales"], 2), ")")
print("Highest profit region:", best_profit["Region"], "($", round(best_profit["Profit"], 2), ")")
print("Best profit margin:", best_margin["Region"], f"({best_margin['Profit_Margin_%']}%)")
print("Lowest profit margin:", worst_margin["Region"], f"({worst_margin['Profit_Margin_%']}%)")

========== REGIONAL PERFORMANCE SUMMARY ==========
Highest sales region: West ($ 725457.82 )
Highest profit region: West ($ 108418.45 )
Best profit margin: West (14.94%)
Lowest profit margin: Central (7.92%)
